In [1]:
from pathlib import Path
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed

import base64
import json
import mimetypes
import os
import re
import time

import matplotlib.pyplot as plt
import pandas as pd
import requests


API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass(
    "OpenRouter API key: "
)

MODEL_FOLDER = Path("..").resolve()
DATASET_FOLDER = MODEL_FOLDER / "ImageDataset"
OUTPUT_FOLDER = MODEL_FOLDER / "FreeModelResults"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

RESPONSES_FILE = OUTPUT_FOLDER / "free_model_responses.csv"
SUMMARY_FILE = OUTPUT_FOLDER / "free_model_summary.csv"
GRAPH_FILE = OUTPUT_FOLDER / "free_model_ranking.png"

MAX_WORKERS = 5

MODELS = {
    "Gemma 4 26B A4B": "google/gemma-4-26b-a4b-it:free",
    "Gemma 4 31B": "google/gemma-4-31b-it:free",
    "Nemotron Nano 12B VL": "nvidia/nemotron-nano-12b-v2-vl:free",
    "Lyria 3 Clip": "google/lyria-3-clip-preview",
    "Lyria 3 Pro": "google/lyria-3-pro-preview",
}

# A separate free model judges the candidates anonymously.
JUDGE_MODEL = (
    "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
)

PROMPT = (
    "Return the answer using exactly two XML tags: "
    "<heading> and <description>. "
    "Inside <heading>, write a short and specific title of 2-6 words "
    "that clearly identifies what is visible in the image. "
    "Inside <description>, identify the main subject and describe it "
    "in one concise, factual paragraph of 2-3 sentences. "
    "Include useful colors, materials, shapes, context and readable text. "
    "Avoid unsupported assumptions. "
    "Use exactly this format: "
    "<heading>Specific title</heading>"
    "<description>Factual description.</description>"
)

JUDGE_PROMPT = """
Evaluate the candidate answer strictly against the attached image.
The candidate model is anonymous.

Return only JSON with these integer fields:
- subject_accuracy: 0 to 4
- visual_details: 0 to 2
- no_hallucinations: 0 to 2
- clarity: 0 to 1
- heading_quality: 0 to 1
- reason: one short sentence

Candidate heading:
{heading}

Candidate description:
{description}
"""

ACCEPTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}


def encode_image(image_path):
    image_bytes = image_path.read_bytes()
    mime_type = (
        mimetypes.guess_type(image_path.name)[0]
        or "image/jpeg"
    )
    image_b64 = base64.b64encode(image_bytes).decode("utf-8")
    return mime_type, image_b64


def request_openrouter(payload, attempts=5):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json",
                },
                json=payload,
                timeout=180,
            )

            if response.status_code in {429, 502, 503, 504}:
                raise requests.HTTPError(
                    f"HTTP {response.status_code}: {response.text[:300]}",
                    response=response,
                )

            response.raise_for_status()
            return response.json()["choices"][0]["message"]["content"]

        except Exception as error:
            last_error = error

            if attempt == attempts:
                break

            time.sleep(attempt * 3)

    raise last_error


def extract_xml(response_text):
    heading_match = re.search(
        r"<heading>(.*?)</heading>",
        response_text,
        re.DOTALL | re.IGNORECASE,
    )
    description_match = re.search(
        r"<description>(.*?)</description>",
        response_text,
        re.DOTALL | re.IGNORECASE,
    )

    heading = heading_match.group(1).strip() if heading_match else ""
    description = (
        description_match.group(1).strip()
        if description_match
        else response_text.strip()
    )
    format_ok = bool(heading_match and description_match)
    return heading, description, format_ok


def generate_description(model_label, model_id, image_path):
    mime_type, image_b64 = encode_image(image_path)

    payload = {
        "model": model_id,
        "temperature": 0,
        "max_tokens": 300,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{image_b64}"
                        },
                    },
                ],
            }
        ],
    }

    started = time.perf_counter()

    try:
        response_text = request_openrouter(payload)
        heading, description, format_ok = extract_xml(response_text)
        status = "success"
        error = ""
    except Exception as exception:
        response_text = ""
        heading = ""
        description = ""
        format_ok = False
        status = "error"
        error = str(exception)

    return {
        "model": model_label,
        "model_id": model_id,
        "image_path": image_path.as_posix(),
        "status": status,
        "error": error,
        "format_ok": format_ok,
        "response_time_seconds": round(
            time.perf_counter() - started,
            2,
        ),
        "heading": heading,
        "description": description,
        "ai_response": response_text,
    }


def judge_description(row):
    image_path = Path(row["image_path"])
    mime_type, image_b64 = encode_image(image_path)

    prompt = JUDGE_PROMPT.format(
        heading=row["heading"],
        description=row["description"],
    )

    payload = {
        "model": JUDGE_MODEL,
        "temperature": 0,
        "max_tokens": 250,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{image_b64}"
                        },
                    },
                ],
            }
        ],
    }

    try:
        content = request_openrouter(payload)
        json_match = re.search(r"\{.*\}", content, re.DOTALL)

        if not json_match:
            raise ValueError(f"Judge returned invalid JSON: {content}")

        scores = json.loads(json_match.group(0))

        subject = min(max(int(scores["subject_accuracy"]), 0), 4)
        details = min(max(int(scores["visual_details"]), 0), 2)
        hallucinations = min(
            max(int(scores["no_hallucinations"]), 0),
            2,
        )
        clarity = min(max(int(scores["clarity"]), 0), 1)
        heading = min(max(int(scores["heading_quality"]), 0), 1)

        return {
            "judge_status": "success",
            "judge_error": "",
            "quality_score": (
                subject
                + details
                + hallucinations
                + clarity
                + heading
            ),
            "judge_reason": scores.get("reason", ""),
        }

    except Exception as exception:
        return {
            "judge_status": "error",
            "judge_error": str(exception),
            "quality_score": None,
            "judge_reason": "",
        }


image_paths = sorted(
    path
    for path in DATASET_FOLDER.iterdir()
    if path.is_file()
    and path.suffix.lower() in ACCEPTED_EXTENSIONS
)

if not image_paths:
    raise ValueError(f"No images found in {DATASET_FOLDER}")

print(f"Testing {len(MODELS)} free models on {len(image_paths)} images.")

generation_jobs = [
    (model_label, model_id, image_path)
    for model_label, model_id in MODELS.items()
    for image_path in image_paths
]

results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            generate_description,
            model_label,
            model_id,
            image_path,
        ): (model_label, image_path)
        for model_label, model_id, image_path in generation_jobs
    }

    for index, future in enumerate(as_completed(futures), start=1):
        model_label, image_path = futures[future]
        result = future.result()
        results.append(result)

        print(
            f"Generation {index}/{len(futures)}: "
            f"{model_label} - {image_path.name} - {result['status']}"
        )

        pd.DataFrame(results).to_csv(
            RESPONSES_FILE,
            index=False,
            encoding="utf-8-sig",
        )


successful_indexes = [
    index
    for index, result in enumerate(results)
    if result["status"] == "success"
]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(judge_description, results[index]): index
        for index in successful_indexes
    }

    for completed, future in enumerate(as_completed(futures), start=1):
        result_index = futures[future]
        results[result_index].update(future.result())

        print(
            f"Judge {completed}/{len(futures)}: "
            f"{results[result_index]['model']} - "
            f"{Path(results[result_index]['image_path']).name}"
        )

        pd.DataFrame(results).to_csv(
            RESPONSES_FILE,
            index=False,
            encoding="utf-8-sig",
        )


for result in results:
    if result["status"] != "success":
        result.update(
            {
                "judge_status": "not_run",
                "judge_error": "Generation failed",
                "quality_score": None,
                "judge_reason": "",
            }
        )

results_df = pd.DataFrame(results)
results_df.to_csv(
    RESPONSES_FILE,
    index=False,
    encoding="utf-8-sig",
)

summary_rows = []

for model_label in MODELS:
    model_results = results_df[results_df["model"].eq(model_label)]
    successful = model_results["status"].eq("success").sum()
    total = len(model_results)
    judged = model_results[model_results["judge_status"].eq("success")]

    quality_score = judged["quality_score"].mean()
    success_rate = successful / total if total else 0

    summary_rows.append(
        {
            "model": model_label,
            "model_id": MODELS[model_label],
            "quality_score": round(quality_score, 2)
            if pd.notna(quality_score)
            else None,
            "success_rate_percent": round(success_rate * 100, 1),
            "format_success_percent": round(
                model_results["format_ok"].fillna(False).mean() * 100,
                1,
            ),
            "average_response_time": round(
                pd.to_numeric(
                    model_results.loc[
                        model_results["status"].eq("success"),
                        "response_time_seconds",
                    ],
                    errors="coerce",
                ).mean(),
                2,
            ),
            "errors": int(total - successful),
            "final_score": round(
                (quality_score if pd.notna(quality_score) else 0)
                * success_rate,
                2,
            ),
        }
    )

summary = pd.DataFrame(summary_rows).sort_values(
    "final_score",
    ascending=False,
)

summary.to_csv(
    SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig",
)

plt.style.use("dark_background")
figure, axis = plt.subplots(figsize=(13, 7))
figure.patch.set_facecolor("#08111f")
axis.set_facecolor("#111d31")

plot_data = summary.sort_values("final_score", ascending=True)
bars = axis.barh(
    plot_data["model"],
    plot_data["final_score"],
    color="#34d399",
)

axis.set_xlim(0, 10.8)
axis.set_xlabel("Final score / 10 (quality penalized by failed requests)")
axis.set_title(
    "OpenLens Free Vision Model Ranking",
    fontsize=19,
    fontweight="bold",
    pad=18,
)
axis.grid(axis="x", linestyle="--", alpha=0.2)
axis.set_axisbelow(True)

for bar, score in zip(bars, plot_data["final_score"]):
    axis.text(
        score + 0.08,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.2f}",
        va="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig(
    GRAPH_FILE,
    dpi=250,
    bbox_inches="tight",
    facecolor=figure.get_facecolor(),
)
plt.close(figure)

print()
print("FREE MODEL RANKING")
print("=" * 70)
print(summary.to_string(index=False))
print()
print(f"Best free model: {summary.iloc[0]['model']}")
print(f"Results: {OUTPUT_FOLDER}")


Testing 5 free models on 10 images.
Generation 1/50: Gemma 4 26B A4B - Blurry_city.jpg - success
Generation 2/50: Gemma 4 26B A4B - crowded_market.jpg - success
Generation 3/50: Gemma 4 26B A4B - camouflaged_animal.jpg - success
Generation 4/50: Gemma 4 26B A4B - cyclist.jpg - success
Generation 5/50: Gemma 4 26B A4B - night_traffic_blur.jpg - success
Generation 6/50: Gemma 4 26B A4B - dark_light_man.jpg - success
Generation 7/50: Gemma 4 26B A4B - distant_road_sign.jpg - success
Generation 8/50: Gemma 4 26B A4B - smoke_ships.jpg - success
Generation 9/50: Gemma 4 26B A4B - rain_window_reflection.jpg - success
Generation 10/50: Gemma 4 26B A4B - insect.jpg - success
Generation 11/50: Gemma 4 31B - Blurry_city.jpg - error
Generation 12/50: Gemma 4 31B - camouflaged_animal.jpg - error
Generation 13/50: Gemma 4 31B - crowded_market.jpg - error
Generation 14/50: Gemma 4 31B - cyclist.jpg - error
Generation 15/50: Gemma 4 31B - dark_light_man.jpg - error
Generation 16/50: Gemma 4 31B - dist